# Severstal Submission-Only Notebook
Notebook ini hanya untuk inferensi dan membuat `submission.csv` tanpa training ulang.

## Dataset yang harus di-attach
1. Severstal competition dataset (berisi `test_images` dan `sample_submission.csv`)
2. Source code dataset (berisi `model.py`, `train_config.py`, `tta.py`)
3. Model checkpoint dataset (berisi `.pth`, misalnya `fpn_efficientnet_b3_best.pth`)

In [1]:
# Cell 1: Resolve paths + load source files
import os
import sys
import shutil
from pathlib import Path

KAGGLE_INPUT = Path('/kaggle/input')
WORKDIR = Path('/kaggle/working')

# Fixed paths from your datasets (preferred)
PREFERRED_COMPETITION_DIR = Path('/kaggle/input/competitions/severstal-steel-defect-detection')
PREFERRED_MODEL_DIR = Path('/kaggle/input/datasets/kurobasgari/best-model-v1')
PREFERRED_MODEL_FILE = Path('/kaggle/input/datasets/kurobasgari/best-model-v1/fpn_efficientnet_b3_best.pth')
PREFERRED_SOURCE_DIR = Path('/kaggle/input/datasets/kurobasgari/source-code')

# Find competition dir (preferred path first)
competition_dir = None
if (PREFERRED_COMPETITION_DIR / 'sample_submission.csv').exists() and (PREFERRED_COMPETITION_DIR / 'test_images').exists():
    competition_dir = PREFERRED_COMPETITION_DIR
else:
    for p in KAGGLE_INPUT.glob('**/sample_submission.csv'):
        if (p.parent / 'test_images').exists():
            competition_dir = p.parent
            break

if competition_dir is None:
    raise FileNotFoundError('Cannot find competition dataset containing sample_submission.csv and test_images')

# Resolve source dir (preferred path first, then fallback scan)
source_dir = None
if (PREFERRED_SOURCE_DIR / 'train_config.py').exists() and (PREFERRED_SOURCE_DIR / 'model.py').exists() and (PREFERRED_SOURCE_DIR / 'tta.py').exists():
    source_dir = PREFERRED_SOURCE_DIR
else:
    for p in KAGGLE_INPUT.glob('**/train_config.py'):
        parent = p.parent
        if (parent / 'model.py').exists() and (parent / 'tta.py').exists():
            source_dir = parent
            break

if source_dir is None:
    raise FileNotFoundError('Cannot find source code dataset (train_config.py/model.py/tta.py)')

# Copy source code to working dir
for py in source_dir.glob('*.py'):
    shutil.copy2(py, WORKDIR / py.name)

sys.path.insert(0, str(WORKDIR))

# Resolve checkpoint (preferred file first, then preferred dir, then fallback scan)
checkpoint = None
if PREFERRED_MODEL_FILE.exists():
    checkpoint = PREFERRED_MODEL_FILE
elif PREFERRED_MODEL_DIR.exists():
    preferred_candidates = sorted(list(PREFERRED_MODEL_DIR.glob('*best*.pth')) + list(PREFERRED_MODEL_DIR.glob('*.pth')))
    if preferred_candidates:
        checkpoint = preferred_candidates[0]

if checkpoint is None:
    candidates = sorted(list(KAGGLE_INPUT.glob('**/*best*.pth')) + list(KAGGLE_INPUT.glob('**/*.pth')), key=lambda x: x.name)
    if candidates:
        checkpoint = candidates[0]

if checkpoint is None:
    raise FileNotFoundError('Cannot find model checkpoint .pth in attached datasets')

print('Competition dir:', competition_dir)
print('Source dir     :', source_dir)
print('Checkpoint     :', checkpoint)
print('Files in /kaggle/working:', sorted([p.name for p in WORKDIR.glob('*.py')]))

Competition dir: /kaggle/input/competitions/severstal-steel-defect-detection
Source dir     : /kaggle/input/datasets/kurobasgari/source-code
Checkpoint     : /kaggle/input/datasets/kurobasgari/best-model-v1/fpn_efficientnet_b3_best.pth
Files in /kaggle/working: ['dataset.py', 'export_onnx.py', 'losses.py', 'model.py', 'predict_test.py', 'train.py', 'train_config.py', 'tta.py']


In [2]:
# Cell 2: HOTFIX dependency untuk mencegah error `smp is not defined`
import importlib.util
import subprocess
import sys
from pathlib import Path


def has_module(name: str) -> bool:
    return importlib.util.find_spec(name) is not None


def find_local_wheels(pkg_name: str):
    patterns = [
        f"**/{pkg_name.replace('-', '_')}-*.whl",
        f"**/{pkg_name}-*.whl",
    ]
    wheels = []
    for pat in patterns:
        wheels.extend(Path('/kaggle/input').glob(pat))
    return sorted(set(wheels), key=lambda p: p.name)


required = {
    'segmentation_models_pytorch': 'segmentation-models-pytorch',
    'timm': 'timm',
}

missing = [pip_name for mod_name, pip_name in required.items() if not has_module(mod_name)]
if missing:
    print('Missing:', missing)

    # 1) online install
    online = subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], capture_output=True, text=True)
    if online.returncode != 0:
        print('Online install gagal, coba wheel offline dari /kaggle/input ...')
        print((online.stderr or online.stdout)[-800:])

        # 2) offline install
        for pkg in missing:
            mod_name = 'segmentation_models_pytorch' if pkg == 'segmentation-models-pytorch' else pkg
            if has_module(mod_name):
                continue
            wheels = find_local_wheels(pkg)
            if not wheels:
                print(f'Wheel tidak ditemukan untuk {pkg}')
                continue
            wheel = wheels[-1]
            print('Installing wheel:', wheel)
            off = subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', str(wheel)], capture_output=True, text=True)
            if off.returncode != 0:
                print((off.stderr or off.stdout)[-800:])

final_missing = [pip_name for mod_name, pip_name in required.items() if not has_module(mod_name)]
if final_missing:
    raise RuntimeError(
        'Dependency belum tersedia: ' + ', '.join(final_missing) +
        '. Attach wheel dataset (segmentation-models-pytorch + timm) atau enable internet.'
    )

import segmentation_models_pytorch as smp
import timm
print('Dependency OK | smp:', smp.__version__, '| timm:', timm.__version__)

Missing: ['segmentation-models-pytorch']
Dependency OK | smp: 0.5.0 | timm: 1.0.26


In [5]:
# Cell 2: Ensure dependencies + imports + helpers
import importlib.util
import subprocess
import sys
from pathlib import Path


def has_module(name: str) -> bool:
    return importlib.util.find_spec(name) is not None


def find_local_wheels(pkg_name: str):
    patterns = [
        f"**/{pkg_name.replace('-', '_')}-*.whl",
        f"**/{pkg_name}-*.whl",
    ]
    wheels = []
    for pat in patterns:
        wheels.extend(Path('/kaggle/input').glob(pat))
    return sorted(set(wheels), key=lambda p: p.name)


required = {
    'segmentation_models_pytorch': 'segmentation-models-pytorch',
    'timm': 'timm',
}

missing = [pip_name for mod_name, pip_name in required.items() if not has_module(mod_name)]
if missing:
    print('Missing dependencies:', missing)

    # Try online install first (if internet available)
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', *missing],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print('Online install failed, trying offline wheels from /kaggle/input ...')
        print((result.stderr or result.stdout)[-800:])

        for pkg in missing:
            if has_module('segmentation_models_pytorch' if pkg == 'segmentation-models-pytorch' else pkg):
                continue
            wheels = find_local_wheels(pkg)
            if not wheels:
                print(f'No wheel found for: {pkg}')
                continue
            wheel = wheels[-1]
            print(f'Installing wheel: {wheel}')
            off = subprocess.run(
                [sys.executable, '-m', 'pip', 'install', '--no-index', str(wheel)],
                capture_output=True,
                text=True,
            )
            if off.returncode != 0:
                print((off.stderr or off.stdout)[-800:])

final_missing = [pip_name for mod_name, pip_name in required.items() if not has_module(mod_name)]
if final_missing:
    raise RuntimeError(
        'Missing dependencies for inference: '
        + ', '.join(final_missing)
        + '. Attach wheel dataset to Kaggle input or enable internet.'
    )

import cv2
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from train_config import INPUT_HEIGHT, INPUT_WIDTH, PIXEL_THRESHOLD, MIN_DEFECT_PIXELS
from model import load_trained_model

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device, '| GPU count:', torch.cuda.device_count())

def preprocess_image(path: Path) -> np.ndarray:
    image = cv2.imread(str(path))
    if image is None:
        raise FileNotFoundError(f'Cannot read image: {path}')
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (INPUT_WIDTH, INPUT_HEIGHT), interpolation=cv2.INTER_LINEAR)
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
    gray = (gray - 0.485) / 0.229
    return gray[None, :, :]

def mask_to_rle(mask: np.ndarray) -> str:
    # Kaggle Severstal format: flatten in Fortran order
    pixels = mask.T.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs)

model = load_trained_model(str(checkpoint), device=device)
print('Model loaded')

Device: cuda | GPU count: 2


config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

Model loaded


In [ ]:
# Cell 3: Inference test_images + build submission.csv
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

sample_sub = pd.read_csv(competition_dir / 'sample_submission.csv')
test_dir = competition_dir / 'test_images'
test_images = sorted(test_dir.glob('*.jpg'))

if not test_images:
    raise FileNotFoundError(f'No test images found in {test_dir}')

print('Test images:', len(test_images))

# Multi-GPU: DataParallel mengparalelkan forward() ke semua GPU
n_gpu = torch.cuda.device_count()
if n_gpu > 1:
    print(f'Using {n_gpu} GPUs via DataParallel')
    dp_model = nn.DataParallel(model)
else:
    print(f'Using 1 GPU')
    dp_model = model


class TestDataset(Dataset):
    def __init__(self, paths):
        self.paths = paths

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        return torch.from_numpy(preprocess_image(self.paths[idx])), self.paths[idx].name


# Tuning: naikkan batch_size & num_workers sesuai GPU/CPU
batch_size = 32 * max(1, n_gpu)  # scale batch_size sesuai jumlah GPU
num_workers = 4

loader = DataLoader(
    TestDataset(test_images),
    batch_size=batch_size,
    num_workers=num_workers,
    pin_memory=(device == 'cuda'),
    prefetch_factor=2 if num_workers > 0 else None,
    persistent_workers=(num_workers > 0),
)

rows = []
use_amp = device == 'cuda'

for batch_t, image_ids in tqdm(loader, desc='Inference'):
    batch_t = batch_t.to(device=device, dtype=torch.float32, non_blocking=True)

    with torch.no_grad():
        with torch.cuda.amp.autocast(enabled=use_amp):
            # DataParallel memparalelkan forward() ke semua GPU
            output = dp_model(batch_t)
        seg_prob = output['seg_logits'].sigmoid()
        cls_prob = output['cls_logits'].sigmoid()
        # Soft gating: sama seperti model.predict()
        masks = (seg_prob * cls_prob[:, :, None, None]).detach().cpu().numpy()  # (B, 4, H, W)

    for b_idx, image_id in enumerate(image_ids):
        for c in range(4):
            m = masks[b_idx, c]
            m = (m > PIXEL_THRESHOLD).astype(np.uint8)
            m = cv2.resize(m, (1600, 256), interpolation=cv2.INTER_NEAREST)

            rle = mask_to_rle(m) if int(m.sum()) >= MIN_DEFECT_PIXELS else ''
            rows.append({'ImageId_ClassId': f'{image_id}_{c+1}', 'EncodedPixels': rle})

submission = pd.DataFrame(rows)
submission = sample_sub[['ImageId_ClassId']].merge(submission, on='ImageId_ClassId', how='left')
submission['EncodedPixels'] = submission['EncodedPixels'].fillna('')

out_path = WORKDIR / 'submission.csv'
submission.to_csv(out_path, index=False)
print('Saved:', out_path)
print(submission.head(8))


Test images: 5506


Inference:   0%|          | 0/345 [00:00<?, ?it/s]

KeyError: "None of [Index(['ImageId_ClassId'], dtype='object')] are in the [columns]"

In [7]:
# Cell 4: Quick sanity checks
print('submission rows:', len(submission))
print('empty masks:', (submission['EncodedPixels'] == '').sum())
print('non-empty masks:', (submission['EncodedPixels'] != '').sum())

assert len(submission) == len(sample_sub), 'Row count mismatch vs sample submission'
print('Submission format OK')

submission rows: 22024
empty masks: 19232
non-empty masks: 2792


AssertionError: Row count mismatch vs sample submission